In [7]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Load environment variables
load_dotenv()

azure_foundry_endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT")
azure_foundry_deployment_name = os.getenv("AZURE_FOUNDRY_DEPLOYMENT_NAME")
azure_deployment_version = os.getenv("AZURE_DEPLOYMENT_VERSION")

endpoint = os.getenv("ENDPOINT_URL", azure_foundry_endpoint)
deployment = os.getenv("DEPLOYMENT_NAME", azure_foundry_deployment_name)
api_version = os.getenv("API_VERSION", azure_deployment_version)

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
    )

client = AzureOpenAI(
    azure_endpoint=endpoint,
    azure_ad_token_provider=token_provider,
    api_version=api_version,
    )

with open("..\\system_prompt.txt", "r", encoding="utf-8") as handle:
    system_prompt = handle.read().strip()

def generate_alt_text(image_url: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe the image in one sentence. Keep it concise and relevant to product"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        },
    ]
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=120,
    )
    return completion.choices[0].message.content

# Example usage
image_url = "https://i8.amplience.net/i/epsonemear/a17839-productpicture-hires-nl-nl-et-1810-1_header_2000x2000?fmt=auto&img404=missing_product&v=1"
alt_text_en = generate_alt_text(image_url)
alt_text_en

'A black Epson printer and 3 toner cartridges against a white background.'

In [8]:
# Translate the generated alt text into multiple languages
language_names = {
    "nl": "Dutch",
    "fr": "French",
    "de": "German",
    "es": "Spanish",
    "it": "Italian",
    "pt": "Portuguese",
    "ja": "Japanese",
    "zh": "Simplified Chinese",
    "sv": "Swedish",
    "da": "Danish",
    "no": "Norwegian",
    "hi": "Hindi",
}

translate_system_prompt = (
    "You are an expert translator specializing in e-commerce product descriptions and alt text.\n\n"
    "Your task is to translate product alt text accurately and concisely, maintaining:\n"
    "1. Exact product name/model (do not translate brand or model numbers)\n"
    "2. Conciseness (max 125 characters)\n"
    "3. Clarity and accessibility for screen reader users\n"
    "4. Suitability for e-commerce context\n\n"
    "Always respond with ONLY the translated text, no additional explanation or quotes."
 )

def translate_alt_text(text: str, lang_code: str) -> str:
    lang_name = language_names.get(lang_code, lang_code.capitalize())
    user_prompt = (
        f"Translate this product alt text to {lang_name}, "
        "keeping it concise (max 125 chars):\n\n"
        f"\"{text}\""
    )
    messages = [
        {"role": "system", "content": translate_system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=200,
    )
    return completion.choices[0].message.content

# Run the next cell to translate into selected languages.

In [9]:
# Select a few languages and translate (for each language code)
target_language_codes = ["nl", "es", "ja"]

translations = {}
translations["en"] = alt_text_en  # Include original English alt text
for code in target_language_codes:
    translations[code] = translate_alt_text(alt_text_en, code)

translations

{'en': 'A black Epson printer and 3 toner cartridges against a white background.',
 'nl': '"Een zwarte Epson printer met 3 toner cartridges tegen een witte achtergrond."',
 'es': '"Ejecutivo de copia en negro para Epson, con 3 carteritas de tinta, fondo blanco."',
 'ja': 'エプソンプリンターと3本の墨替え用トナー、白背景.'}